In [ ]:
# --- combined_script_with_kfold_xgb_optimized_and_submission.py ---
# Script completo:
# 1. Carregar Dados
# 2. Mapear/Codificar Target
# 3. OTIMIZAR HIPERPARÂMETROS XGBOOST (com GridSearchCV) <-- NOVO
# 4. Definir K-Fold e Feature Engineering
# 5. Executar K-Fold CV (RF + XGB) para avaliação E OBTENÇÃO DE PREVISÕES OOF
# 6. OTIMIZAR PESOS DE BLENDING (com previsões OOF) <-- NOVO
# 7. Apresentar resultados da CV e pesos ótimos
# 8. Treinar modelos finais (RF + XGB) em TODOS os dados de treino
# 9. ANALISAR FEATURE IMPORTANCE <-- NOVO
# 10. Processar dados de teste
# 11. Fazer Blending (RF + XGB) nas previsões de teste com pesos otimizados
# 12. Gerar ficheiro de submissão final

import pandas as pd
import numpy as np
import holidays
import os
import time  # Para medir o tempo

# Modelos e Ferramentas Sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedKFold, GridSearchCV # Adicionar GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
# %pip install --upgrade xgboost
# Modelo XGBoost
import xgboost as xgb

# Visualização (Opcional, para Feature Importance)
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configurações ---
RANDOM_STATE = 2020
N_SPLITS = 5      # Número de folds para Cross-Validation
INITIAL_WEIGHT_RF = 0.50  # Peso inicial para Blending (será otimizado)
INITIAL_WEIGHT_XGB = 0.50 # Peso inicial para Blending (será otimizado)
N_TOP_FEATURES = 20 # Quantas features mostrar na análise de importância

# Verificar se os pesos iniciais somam 1
assert np.isclose(INITIAL_WEIGHT_RF + INITIAL_WEIGHT_XGB, 1.0), "Os pesos iniciais devem somar 1.0"

# --- Melhores Hiperparâmetros (RF fixo, XGB será otimizado) ---
best_params_rf = {
    'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 2,
    'min_samples_split': 3, 'n_estimators': 1000,
    'class_weight': 'balanced', # Garantir balanceamento
    'random_state': RANDOM_STATE,
    'n_jobs': -1
}

# Parâmetros *BASE* e *GRID* para otimização do XGBoost
base_params_xgb = {
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',
    'use_label_encoder': False,
    'random_state': RANDOM_STATE
}

param_grid_xgb = {
    'n_estimators': [1000], # Aumentar um pouco
    'learning_rate': [0.05],   # Testar taxas diferentes
    'max_depth': [8],         # Profundidade
    'min_child_weight': [1],     # Regularização
    'gamma': [0.1],            # Regularização
    'subsample': [0.8],        # Amostragem de linhas
    'colsample_bytree': [0.9]  # Amostragem de colunas
}
# Manter best_params_xgb como um dicionário que será preenchido
best_params_xgb = base_params_xgb.copy()

print("--- INÍCIO DO PROCESSO COMPLETO (OTIMIZAÇÃO + CV + SUBMISSÃO com XGB) ---")
start_time_total = time.time()

# === PASSO 1: CARREGAR DADOS ===
print("\n[1/12] A carregar os dados...")
try:
    dfTrain_orig = pd.read_csv('training_data.csv', encoding='latin1')
    dfTest_orig = pd.read_csv('test_data.csv', encoding='latin1')
except FileNotFoundError as e:
    print(f"Erro: Ficheiro de dados não encontrado - {e}.")
    exit()

# === PASSO 2: MAPEAMENTO/CODIFICAÇÃO INICIAL DO TARGET ===
print("[2/12] A mapear e codificar o target...")
mappingSpeedDiff_orig = {'Low': 0, 'Medium': 1, 'High': 2, 'Very_High': 3}
# Target original para RF e StratifiedKFold (-1 a 3)
dfTrain_orig['AVERAGE_SPEED_DIFF_Mapped_RF'] = dfTrain_orig['AVERAGE_SPEED_DIFF'].map(mappingSpeedDiff_orig).fillna(-1)
y_orig_for_split = dfTrain_orig['AVERAGE_SPEED_DIFF_Mapped_RF']

# Target para XGBoost (0 a 4) - Mapear e Codificar com LabelEncoder
y_map_for_xgb = dfTrain_orig['AVERAGE_SPEED_DIFF_Mapped_RF'].map({-1: 0, 0: 1, 1: 2, 2: 3, 3: 4})
le = LabelEncoder()
y_xgb_target_encoded_full = le.fit_transform(y_map_for_xgb)
n_classes = len(le.classes_)
print(f"Target remapeado para XGB (0 a {n_classes-1}). Classes originais mapeadas: {dict(zip(le.classes_, le.inverse_transform(le.classes_)))}")
target_names_report = ['None', 'Low', 'Medium', 'High', 'Very_High'] # Baseado na ordem 0-4

# Atualizar base_params_xgb com num_class correto
base_params_xgb['num_class'] = n_classes
best_params_xgb['num_class'] = n_classes # Garantir que está nos params finais também

# === PASSO 3: FUNÇÃO DE FEATURE ENGINEERING ===
print("[3/12] A definir função de Feature Engineering...")
# (Função definida aqui para ser usada na otimização e no K-Fold)
def engineer_features(df_orig):
    df = df_orig.copy()
    df['record_date'] = pd.to_datetime(df['record_date'])
    df['hour'] = df['record_date'].dt.hour
    df['dayOfWeek'] = df['record_date'].dt.dayofweek
    df['month'] = df['record_date'].dt.month
    df['year'] = df['record_date'].dt.year
    df['dayOfYear'] = df['record_date'].dt.dayofyear # Nova feature
    df['weekOfYear'] = df['record_date'].dt.isocalendar().week.astype(int) # Nova feature

    df['isRushHour'] = ((df['hour'] >= 7) & (df['hour'] <= 10) |
                        (df['hour'] >= 16) & (df['hour'] <= 21)).astype(int)
    df['IS_WEEKEND'] = (df['dayOfWeek'] >= 5).astype(int)
    try:
        anos = df['year'].unique()
        feriados_portugal = holidays.Portugal(years=anos)
        df['is_holiday'] = df['record_date'].dt.date.isin(feriados_portugal).astype(int)
    except NameError:
        df['is_holiday'] = 0

    # Features Cíclicas
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['dayOfWeek_sin'] = np.sin(2 * np.pi * df['dayOfWeek'] / 7.0)
    df['dayOfWeek_cos'] = np.cos(2 * np.pi * df['dayOfWeek'] / 7.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0) # Nova feature
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0) # Nova feature

    # Ratios e Tratamento de Divisão por Zero
    df['AVERAGE_FREE_FLOW_TIME'] = df['AVERAGE_FREE_FLOW_TIME'].replace(0, 1e-6)
    df['TIME_DELAY_RATIO'] = df['AVERAGE_TIME_DIFF'] / df['AVERAGE_FREE_FLOW_TIME']
    sum_time = df['AVERAGE_FREE_FLOW_TIME'] + df['AVERAGE_TIME_DIFF']
    df['FREE_FLOW_RATIO'] = df['AVERAGE_FREE_FLOW_TIME'] / sum_time.replace(0, 1e-6)
    df.replace([np.inf, -np.inf], 0, inplace=True)

    # Mapeamentos Categóricos (Clima)
    mappingLuminosity = {'DARK': 0, 'LOW_LIGHT': 1, 'LIGHT': 2}
    mappingCloudiness = {'céu limpo': 0, 'céu claro': 0, 'céu pouco nublado': 1, 'algumas nuvens': 1, 'nuvens dispersas': 2, 'nuvens quebradas': 3, 'nuvens quebrados': 3, 'nublado': 4, 'tempo nublado': 4}
    mappingRain = {'chuvisco fraco': 0, 'chuvisco e chuva fraca': 1, 'chuva fraca': 1, 'chuva leve': 1, 'aguaceiros fracos': 2, 'chuva': 2, 'aguaceiros': 3, 'chuva moderada': 3, 'chuva forte': 4, 'chuva de intensidade pesada': 5, 'chuva de intensidade pesado': 5, 'trovoada com chuva leve': 5, 'trovoada com chuva': 6}
    df['LUMINOSITY'] = df['LUMINOSITY'].map(mappingLuminosity).fillna(-1)
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].map(mappingCloudiness).fillna(-1)
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].map(mappingRain).fillna(-1)

    # Feature Tempo Aulas
    p1_inicio = pd.to_datetime('2018-09-12'); p1_fim = pd.to_datetime('2018-12-14')
    p2_inicio = pd.to_datetime('2019-01-03'); p2_fim = pd.to_datetime('2019-04-05')
    p3_inicio = pd.to_datetime('2019-04-23'); p3_fim = pd.to_datetime('2019-06-21')
    cond_p1 = (df['record_date'] >= p1_inicio) & (df['record_date'] <= p1_fim)
    cond_p2 = (df['record_date'] >= p2_inicio) & (df['record_date'] <= p2_fim)
    cond_p3 = (df['record_date'] >= p3_inicio) & (df['record_date'] <= p3_fim)
    df['TEMPO_AULAS'] = (cond_p1 | cond_p2 | cond_p3).astype(int)

    # Novas Features de Interação
    df['rush_hour_weekend'] = df['isRushHour'] * df['IS_WEEKEND']
    df['lumi_x_delay'] = df['LUMINOSITY'] * df['TIME_DELAY_RATIO']
    df['temp_x_humidity'] = df['AVERAGE_TEMPERATURE'] * df['AVERAGE_HUMIDITY']
    df['delay_x_rush'] = df['TIME_DELAY_RATIO'] * df['isRushHour']

    # Mapear Target (se existir) - JÁ FEITO FORA DA FUNÇÃO, mas mantemos por segurança
    if 'AVERAGE_SPEED_DIFF' in df.columns:
         df['AVERAGE_SPEED_DIFF_Mapped_RF'] = df['AVERAGE_SPEED_DIFF'].map(mappingSpeedDiff_orig).fillna(-1)

    # Fill NA final
    df.fillna(0, inplace=True)

    # Colunas a remover
    colunas_a_remover = [
        'AVERAGE_SPEED_DIFF',
        'AVERAGE_SPEED_DIFF_Mapped_RF',
        'record_date',
        'city_name',
        'is_holiday',
        'AVERAGE_TEMPERATURE',
        'AVERAGE_HUMIDITY',
        'AVERAGE_RAIN',
        'AVERAGE_PRECIPITATION',
        'LUMINOSITY',
        'dayOfYear',
        'dayOfWeek',
        'dayOfWeek_cos',
        'dayOfWeek_sin',
        'weekOfYear'
        'AVERAGE_WIND_SPEED',
        'AVERAGE_CLOUDINESS',
        'AVERAGE_WIND_SPEED',
        'AVERAGE_FREE_FLOW_TIME',
        'AVERAGE_ATMOSP_PRESSURE',
        'year',
        'hour',
        'month',
        'day',
        'temp_x_humidity',

    ]
    X_df = df.drop(columns=colunas_a_remover, errors='ignore')

    # Guardar targets se existirem
    targets = {}
    if 'AVERAGE_SPEED_DIFF_Mapped_RF' in df.columns:
        targets['y_rf'] = df['AVERAGE_SPEED_DIFF_Mapped_RF']
        # Mapear e codificar target XGBoost
        targets['y_xgb'] = le.transform(df['AVERAGE_SPEED_DIFF_Mapped_RF'].map({-1: 0, 0: 1, 1: 2, 2: 3, 3: 4}))

    return X_df, targets

# === PASSO 4: OTIMIZAÇÃO DE HIPERPARÂMETROS XGBOOST ===
print("\n[4/12] A otimizar hiperparâmetros do XGBoost (pode demorar)...")
start_time_xgb_tune = time.time()
X_train_full_eng_for_tune, _ = engineer_features(dfTrain_orig)
# y_xgb_target_encoded_full já calculado no Passo 2

xgb_clf_tune = xgb.XGBClassifier(**base_params_xgb) # Usar parâmetros base

grid_search_xgb = GridSearchCV(
    estimator=xgb_clf_tune,
    param_grid=param_grid_xgb,
    cv=2, # Usar 3 folds para rapidez na otimização
    scoring='f1_weighted', # Ou 'f1_weighted'
    n_jobs=-1,
    verbose=1
)

grid_search_xgb.fit(X_train_full_eng_for_tune, y_xgb_target_encoded_full)

# Atualizar best_params_xgb com os resultados do GridSearch
best_params_xgb.update(grid_search_xgb.best_params_)

end_time_xgb_tune = time.time()
print(f"Otimização XGBoost concluída em {(end_time_xgb_tune - start_time_xgb_tune)/60:.2f} minutos.")
print("Melhores parâmetros encontrados para XGBoost:")
print(best_params_xgb)
print(f"Melhor score de CV (Accuracy) na otimização: {grid_search_xgb.best_score_:.4f}")

# === PASSO 5: DEFINIR K-FOLD E LISTAS PARA RESULTADOS/OOF ===
print("\n[5/12] A definir K-Fold para avaliação...")
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_accuracy_scores = []
fold_f1_weighted_scores = []
fold_f1_macro_scores = []
# Listas para guardar previsões Out-of-Fold (OOF) para otimização de pesos
oof_rf_probas = []
oof_xgb_probas = []
oof_y_true = []
oof_indices = []

# === PASSO 6: LOOP DE CROSS-VALIDATION (AVALIAÇÃO E PREVISÕES OOF) ===
print(f"\n[6/12] A iniciar Cross-Validation com {N_SPLITS} folds...")
start_time_cv = time.time()
fold_num = 1
for train_index, val_index in kf.split(dfTrain_orig, y_orig_for_split):
    print(f"\n--- Fold {fold_num}/{N_SPLITS} ---")
    df_fold_train = dfTrain_orig.iloc[train_index]
    df_fold_val = dfTrain_orig.iloc[val_index]

    print("  A aplicar feature engineering...")
    X_fold_train_eng, targets_train = engineer_features(df_fold_train)
    X_fold_val_eng, targets_val = engineer_features(df_fold_val)
    X_fold_train, X_fold_val = X_fold_train_eng.align(X_fold_val_eng, join='inner', axis=1, fill_value=0)

    y_rf_fold_train = targets_train['y_rf']
    y_xgb_fold_train = targets_train['y_xgb']
    y_rf_fold_val = targets_val['y_rf']
    y_xgb_fold_val = targets_val['y_xgb']

    # --- 6.1 Treinar e Prever com RandomForest ---
    print("  A treinar RandomForest...")
    rf_model_fold = RandomForestClassifier(**best_params_rf) # best_params_rf já inclui tudo
    rf_model_fold.fit(X_fold_train, y_rf_fold_train)
    print("  A prever probabilidades OOF com RF...")
    rf_probas_val = rf_model_fold.predict_proba(X_fold_val)
    rf_classes_map = {cls_val: idx for idx, cls_val in enumerate(rf_model_fold.classes_)}
    rf_probas_val_ordered = np.zeros((rf_probas_val.shape[0], n_classes))
    target_rf_order = [-1, 0, 1, 2, 3]
    for i, cls_val in enumerate(target_rf_order):
        if cls_val in rf_classes_map:
            rf_probas_val_ordered[:, i] = rf_probas_val[:, rf_classes_map[cls_val]]

    # --- 6.2 Treinar e Prever com XGBoost ---
    print("  A treinar XGBoost...")
    sample_weights_fold = compute_sample_weight(class_weight='balanced', y=y_xgb_fold_train)
    xgb_model_fold = xgb.XGBClassifier(**best_params_xgb) # Usar os params otimizados
    # Usar early stopping no treino do fold para eficiência
    eval_set = [(X_fold_val, y_xgb_fold_val)]
    xgb_model_fold.fit(X_fold_train, y_xgb_fold_train,
                       sample_weight=sample_weights_fold,
                       eval_set=eval_set,
                       verbose=False)
    print("  A prever probabilidades OOF com XGB...")
    xgb_probas_val = xgb_model_fold.predict_proba(X_fold_val)

    # --- 6.3 Blending e Avaliação ---
    print("  A fazer blending e a avaliar...")
    if rf_probas_val_ordered.shape != xgb_probas_val.shape:
         print(f"    Erro Shape Fold {fold_num}: RF {rf_probas_val_ordered.shape} != XGB {xgb_probas_val.shape}")
         if xgb_probas_val.shape[1] < rf_probas_val_ordered.shape[1]:
             temp_xgb = np.zeros_like(rf_probas_val_ordered)
             temp_xgb[:,:xgb_probas_val.shape[1]] = xgb_probas_val
             xgb_probas_val = temp_xgb; print("     Shape XGB ajustado.")
         else: fold_num += 1; del xgb_model_fold, rf_model_fold; continue

    # Blending com pesos INICIAIS para avaliação do fold
    blended_probas_val = (INITIAL_WEIGHT_RF * rf_probas_val_ordered) + (INITIAL_WEIGHT_XGB * xgb_probas_val)
    blended_preds_numeric = np.argmax(blended_probas_val, axis=1)

    acc = accuracy_score(y_xgb_fold_val, blended_preds_numeric)
    f1_w = f1_score(y_xgb_fold_val, blended_preds_numeric, average='weighted', zero_division=0)
    f1_m = f1_score(y_xgb_fold_val, blended_preds_numeric, average='macro', zero_division=0)
    fold_accuracy_scores.append(acc); fold_f1_weighted_scores.append(f1_w); fold_f1_macro_scores.append(f1_m)
    print(f"    Fold Accuracy: {acc:.4f}, F1 Weighted: {f1_w:.4f}, F1 Macro: {f1_m:.4f}")
    # print("\n    Relatório de Classificação do Fold:") # Opcional, muito verbose
    # print(classification_report(y_xgb_fold_val, blended_preds_numeric, target_names=target_names_report, labels=range(n_classes), zero_division=0))

    # Guardar previsões OOF e targets para otimização de pesos
    oof_rf_probas.append(rf_probas_val_ordered)
    oof_xgb_probas.append(xgb_probas_val)
    oof_y_true.append(y_xgb_fold_val)
    oof_indices.append(val_index) # Guardar índices para garantir a ordem

    fold_num += 1
    del xgb_model_fold, rf_model_fold

end_time_cv = time.time()
print(f"\nCross-Validation concluída em {(end_time_cv - start_time_cv)/60:.2f} minutos.")

# === PASSO 7: OTIMIZAÇÃO DOS PESOS DE BLENDING ===
print("\n[7/12] A otimizar pesos de blending usando previsões OOF...")

# Concatenar resultados OOF na ordem correta
oof_indices = np.concatenate(oof_indices)
oof_y_true = np.concatenate(oof_y_true)
oof_rf_probas = np.concatenate(oof_rf_probas)
oof_xgb_probas = np.concatenate(oof_xgb_probas)

# Reordenar de acordo com os índices originais para garantir
order = np.argsort(oof_indices)
oof_y_true = oof_y_true[order]
oof_rf_probas = oof_rf_probas[order]
oof_xgb_probas = oof_xgb_probas[order]

best_oof_f1 = -1
best_weight_rf_oof = INITIAL_WEIGHT_RF # Manter os default caso nada melhore
best_weight_xgb_oof = INITIAL_WEIGHT_XGB

for w_rf_test in np.arange(0.0, 1.05, 0.05): # Testar pesos de 0 a 1
    w_xgb_test = 1.0 - w_rf_test
    blended_oof_probas = (w_rf_test * oof_rf_probas) + (w_xgb_test * oof_xgb_probas)
    blended_oof_preds = np.argmax(blended_oof_probas, axis=1)
    current_oof_f1 = f1_score(oof_y_true, blended_oof_preds, average='weighted', zero_division=0)
    # print(f"  Testando RF={w_rf_test:.2f}, XGB={w_xgb_test:.2f} -> F1 (OOF): {current_oof_f1:.4f}")
    if current_oof_f1 > best_oof_f1:
        best_oof_f1 = current_oof_f1
        best_weight_rf_oof = w_rf_test
        best_weight_xgb_oof = 1.0 - w_rf_test

# Usar os pesos otimizados para o blending final
WEIGHT_RF = best_weight_rf_oof
WEIGHT_XGB = best_weight_xgb_oof
print(f"Pesos de blending otimizados: RF={WEIGHT_RF:.2f}, XGB={WEIGHT_XGB:.2f} (Melhor F1 OOF: {best_oof_f1:.4f})")

# === PASSO 8: TREINAR MODELOS FINAIS EM TODOS OS DADOS DE TREINO ===
print("\n[8/12] A treinar modelos finais com TODOS os dados de treino...")

print("  A aplicar feature engineering aos dados de treino completos...")
X_train_full_eng, targets_train_full = engineer_features(dfTrain_orig)
y_rf_train_full = targets_train_full['y_rf']
y_xgb_train_full = targets_train_full['y_xgb'] # Target 0-4

print("  A treinar RandomForest final...")
final_model_rf = RandomForestClassifier(**best_params_rf) # Usar params atualizados
final_model_rf.fit(X_train_full_eng, y_rf_train_full)
print("  RandomForest final treinado.")

print("  A treinar XGBoost final...")
sample_weights_full = compute_sample_weight(class_weight='balanced', y=y_xgb_train_full)
final_model_xgb = xgb.XGBClassifier(**best_params_xgb) # Usar params otimizados
# Para o treino final, podemos usar um eval_set (se tivermos um holdout) ou treinar por N rounds
# Vamos treinar com os n_estimators definidos nos best_params_xgb
final_model_xgb.fit(X_train_full_eng, y_xgb_train_full, sample_weight=sample_weights_full, verbose=False)
print("  XGBoost final treinado.")

# === PASSO 9: ANÁLISE DE FEATURE IMPORTANCE ===
print("\n[9/12] A analisar Feature Importance dos modelos finais...")

try:
    # RandomForest
    importances_rf = pd.Series(final_model_rf.feature_importances_, index=X_train_full_eng.columns)
    importances_rf = importances_rf.sort_values(ascending=False)
    print(f"\n--- Top {N_TOP_FEATURES} Features - RandomForest ---")
    print(importances_rf.head(N_TOP_FEATURES))

    # XGBoost
    importances_xgb = pd.Series(final_model_xgb.feature_importances_, index=X_train_full_eng.columns)
    importances_xgb = importances_xgb.sort_values(ascending=False)
    print(f"\n--- Top {N_TOP_FEATURES} Features - XGBoost ---")
    print(importances_xgb.head(N_TOP_FEATURES))

    # Plot (Opcional)
    plt.figure(figsize=(12, 10))
    plt.subplot(2, 1, 1)
    sns.barplot(x=importances_rf.head(N_TOP_FEATURES).values, y=importances_rf.head(N_TOP_FEATURES).index)
    plt.title(f'Top {N_TOP_FEATURES} Feature Importances (RandomForest)')
    plt.xlabel('Importance')
    plt.ylabel('Features')

    plt.subplot(2, 1, 2)
    sns.barplot(x=importances_xgb.head(N_TOP_FEATURES).values, y=importances_xgb.head(N_TOP_FEATURES).index)
    plt.title(f'Top {N_TOP_FEATURES} Feature Importances (XGBoost)')
    plt.xlabel('Importance')
    plt.ylabel('Features')

    plt.tight_layout()
    plt.savefig('feature_importances.png') # Salvar o gráfico
    print("\nGráfico de Feature Importance salvo como 'feature_importances.png'")
    # plt.show() # Descomentar para mostrar o gráfico interativamente

except Exception as e:
    print(f"Erro ao gerar feature importance: {e}")


# === PASSO 10: PROCESSAR DADOS DE TESTE ===
print("\n[10/12] A processar dados de teste...")
X_test_eng, _ = engineer_features(dfTest_orig)
X_train_full_aligned, X_test_aligned = X_train_full_eng.align(X_test_eng, join='inner', axis=1, fill_value=0)

# === PASSO 11: PREVER PROBABILIDADES NO TESTE E FAZER BLENDING ===
print("\n[11/12] A prever probabilidades nos dados de teste...")
print("  A prever com RF final...")
rf_probas_test = final_model_rf.predict_proba(X_test_aligned)
rf_classes_map_final = {cls_val: idx for idx, cls_val in enumerate(final_model_rf.classes_)}
rf_probas_test_ordered = np.zeros((rf_probas_test.shape[0], n_classes))
target_rf_order = [-1, 0, 1, 2, 3]
for i, cls_val in enumerate(target_rf_order):
    if cls_val in rf_classes_map_final:
        rf_probas_test_ordered[:, i] = rf_probas_test[:, rf_classes_map_final[cls_val]]

print("  A prever com XGB final...")
xgb_probas_test = final_model_xgb.predict_proba(X_test_aligned)

print(f"  A fazer blending das previsões de teste com pesos otimizados (RF={WEIGHT_RF:.2f}, XGB={WEIGHT_XGB:.2f})...")
if rf_probas_test_ordered.shape != xgb_probas_test.shape:
    print(f"    Erro Final Blending: Shape RF {rf_probas_test_ordered.shape} != Shape XGB {xgb_probas_test.shape}")
    if xgb_probas_test.shape[1] < rf_probas_test_ordered.shape[1]:
        temp_xgb = np.zeros_like(rf_probas_test_ordered)
        temp_xgb[:,:xgb_probas_test.shape[1]] = xgb_probas_test
        xgb_probas_test = temp_xgb; print("     Shape XGB ajustado.")
    else: print("    Erro fatal. A sair."); exit()

blended_probas_test = (WEIGHT_RF * rf_probas_test_ordered) + (WEIGHT_XGB * xgb_probas_test)
final_preds_test_numeric = np.argmax(blended_probas_test, axis=1) # Índices 0-4

# Mapear para labels de texto
reverse_map_final = {0: 'None', 1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very_High'}
final_preds_test_labels = [reverse_map_final[pred] for pred in final_preds_test_numeric]

# === PASSO 12: GERAR FICHEIRO DE SUBMISSÃO ===
print("\n[12/12] A gerar ficheiro de submissão final...")
row_ids = range(1, len(dfTest_orig) + 1)
output_filename = f'submission_blend_RF{int(WEIGHT_RF*100)}_XGB{int(WEIGHT_XGB*100)}_final_opt.csv'
output = pd.DataFrame({'RowId': row_ids, 'Speed_Diff': final_preds_test_labels})
output.to_csv(output_filename, index=False)

end_time_total = time.time()
print(f"\n--- FIM DO PROCESSO (Total: {(end_time_total - start_time_total)/60:.2f} minutos) ---")
print(f"Ficheiro de submissão final '{output_filename}' criado com sucesso!")
print("\nDistribuição das classes na submissão (Contagem):")
print(output['Speed_Diff'].value_counts().sort_index())
print("\nDistribuição das classes na submissão (Percentagem):")
print(output['Speed_Diff'].value_counts(normalize=True).sort_index())

# Limpar memória
del final_model_xgb
del final_model_rf